<a href="https://colab.research.google.com/github/rafaellopesdesa/hnsbi-toolkit/blob/main/examples/notebooks/hybrid_reference_flow_and_density_ratios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Native reference flow and density ratios

This notebook is the YAML-driven replacement for the original Exercise 5 workflow. It uses only `hnsbi-toolkit`: the ratio trainer, diagnostics, ONNX export, workspace writer, JAX likelihood, and Minuit inference are all native.

In [1]:
from pathlib import Path
import os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/hsbi-toolkit')
    REPO = ROOT / 'hnsbi-toolkit'
    ROOT.mkdir(parents=True, exist_ok=True)
    if not REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/rafaellopesdesa/hnsbi-toolkit.git', str(REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[lhc,flows]'], check=True)
else:
    REPO = Path.cwd()
    if not (REPO / 'pyproject.toml').exists():
        REPO = Path.cwd().parents[1]
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'examples' / 'lhc_analysis'))
print(REPO)


Mounted at /content/drive
/content/drive/MyDrive/hsbi-toolkit/hnsbi-toolkit


## Generate the configured samples

The shared generator supplies signal, background, and reference samples plus response, resolution, and signal-theory variations. Increase the event counts for publication runs.

In [2]:
from generate_distributions import generate
from hnsbi import Project

EXAMPLE = REPO / 'examples' / 'lhc_analysis'
generate(EXAMPLE / 'data', signal_events=120_000, background_events=300_000, reference_events=400_000)
project = Project.load(EXAMPLE / 'analysis.yaml')
print(project.config.features)


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Train the reference and native ratio ensembles

Each ratio member has independent class normalization, deterministic train/validation/holdout splits, embedded preprocessing in ONNX, parity checks, and loss/overtraining/calibration/reweighting/normalization diagnostics.

In [ ]:
reference_artifacts = project.train_reference()
reference = reference_artifacts.training.flow
ratio_artifacts = project.train_ratios(reference, normalization_events=40_000, seed=20260729)
for sample, training in ratio_artifacts.training.items():
    print(sample, training.manifest_path)
    print(training.members[0].metadata.get('diagnostics'))


## Systematics, Asimov closure, workspace, and fit

The Asimov weights use sample-wise $E_q[r_k]$ normalization. The workspace is JSON, while the first user interface remains YAML.

In [ ]:
from hnsbi.inference import MinuitInference

systematic_training = project.train_systematics()
runtime_systematics = project.build_runtime_systematics(systematic_training)
asimov = project.build_configured_asimov(reference=reference, ratios=ratio_artifacts.evaluators, normalizer=ratio_artifacts.normalizer, systematics=runtime_systematics)
workspace = project.write_configured_workspace(asimov, reference_manifest=reference_artifacts.checkpoint_manifest, ratio_manifests={name: value.manifest_path for name, value in ratio_artifacts.training.items()})
likelihood = project.workspace_runtime(workspace.path)
fit = MinuitInference(likelihood).fit()
print('raw count / ESS:', asimov.raw_count, asimov.ess)
print(fit.point, fit.errors)
